# 17j — Frozen-contact (unconditional) vs conditional forecast

The 8j/9j forecast is **conditional** (the "contact-updated iterate", `inst/3` §8). At horizon `h`
its Stage-1 chain is fitted on contacts up to **t₀+h**, its Stage-2 fit pairs fit week `k` with
contacts from week `k+h`, and the NGM is frozen at the **t₀+h** contact matrix. In other words, it
is told the future contacts.

This notebook scores the plain **frozen-contact** counterpart fitted in 8j §2b:
- one Stage-1 chain per (degree × origin) on contacts up to **t₀ only** (`FROZEN_H = 0`, the 8 fit weeks);
- one Stage-2 fit per (combo × origin), with same-week pairing;
- every horizon iterated with `C*`, and antibody, frozen at t₀ (`two_stage_forecast(…; frozen = true)`).

Everything else is the same code path: the per-draw generation interval, the observation-noise
seeds and the median lag plug. **The two schemes therefore differ only in which contacts they
were given.**

- §1 Assemble the frozen forecast (own cache `17j_assembly_frozen_*`) and READ 9j's conditional cache.
- §2 Score both with WIS (`score_wis`, log scale headline).
- §3 9j's figures for the frozen scheme.
- §4 Conditional vs frozen comparison.
- §5 Verification.
- §6 Six scenarios: the variant-specific renewal (9j §V) × contact data, conditional and frozen.

⚠ **A pure reader.** It cannot refit: cell 3 asserts that every `_h0` artefact exists before anything
can reach `fit_or_load_stage2`, and the conditional side comes from `load_9j_assembly`, which never
fits. Outputs go to `res/17j_*` only.

⚠ **The null model is the noise floor.** Its `C*` is the same constant in both schemes, so its
frozen/conditional ratio measures pure Monte-Carlo and Stage-2-refit noise. Read the other models'
gaps against it.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")
using Random, Statistics
mkpath("../res")
default_plot_setting()
T_START = time()   # §5 (5) checks nothing outside res/17j_* was written after this

In [ ]:
# MUST match the 8j run (same token, same origins) — as in 9j. Default = the generation on disk.
STAGE1_USE_NUTS = get(ENV, "STAGE1_USE_NUTS", "true") == "true"
cfg  = FrameworkConfig(constant_contacts = false, stage1_use_nuts = STAGE1_USE_NUTS)
grid = cis_age_grid()

FIT_END = Date(get(ENV, "FIT_END", "2022-01-24"))
raw  = load_raw_contact_inputs()
inf  = load_raw_infection_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw,
                                              origin_max = FIT_END)
haskey(ENV, "ORIGIN_MIN") && filter!(>=(Date(ENV["ORIGIN_MIN"])), FORECAST_ORIGINS)
@assert !isempty(FORECAST_ORIGINS) "ORIGIN_MIN/FIT_END selected no origins"
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

# The same six combos as 8j/9j, in the same order (colours key on it).
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(),   DiagonalMeanNGM()),      # no-interaction
               (NoContactDegree(), NullNGM())])             # null
include("9j_viz_utils.jl")
include("15j_viz_utils.jl")    # load_9j_assembly — the read-only conditional reader
include("17j_viz_utils.jl")
SAVE_DPI[] = parse(Int, get(ENV, "SAVE_DPI", "300"))   # high-resolution PNG export (save_show; display unchanged)
model_labels = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols   = [:steelblue, :darkorange, :seagreen, :purple, :firebrick, :grey40]   # = 9j
println("origins: ", length(wins), " (", first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS),
        ") | token ", contacts_label(cfg), " | frozen h = ", FROZEN_H)

In [ ]:
# ⚠ GUARD — before ANY assembly call. A missing h0 artefact would otherwise be FITTED on demand.
n_h0 = assert_h0_artefacts(wins, combos, cfg)
println("frozen-contact artefacts present: ", n_h0)

## §1 Assembly

In [ ]:
# FROZEN: built (once) by 17j into its OWN cache — never 9j's default path.
REBUILD_FROZEN = false
asm_f = assemble_or_load_forecasts(wins, combos, cfg; grid = grid, raw = raw,
                                   save_dir = "../dt_intermediate",
                                   cache_path = frozen_cache_path(cfg),
                                   rebuild = REBUILD_FROZEN, frozen = true)
@assert isempty(asm_f.skipped) "frozen assembly skipped $(length(asm_f.skipped)) origin×combo: $(asm_f.skipped)"

# CONDITIONAL: 9j's cache, READ ONLY. It may cover more origins than this run (e.g. a smoke run
# over 3 origins) — restrict it to ours, and require that it covers all of them.
asm_c0 = load_9j_assembly(cfg; labels = model_labels)
origins = [w.origin for w in wins]
@assert issubset(origins, asm_c0.origins) "9j's conditional cache does not cover these origins: $(setdiff(origins, asm_c0.origins))"
qall_c = @subset(asm_c0.qall, in.(:forecast_date, Ref(Set(origins))))
fc_c   = Dict(k => v for (k, v) in asm_c0.fc_store if k[1] in Set(origins))
qall_f, fc_f = asm_f.qall, asm_f.fc_store
println("quantile rows — conditional ", nrow(qall_c), ", frozen ", nrow(qall_f))
@assert nrow(qall_c) == nrow(qall_f) "the two schemes do not cover the same units"

## §2 Scoring (WIS, both scales; log scale is the headline)

In [ ]:
scores_c = score_wis(qall_c)
scores_f = score_wis(qall_f)
cmp      = scheme_compare_frames(scores_c, scores_f)
ci_frame = paired_ratio_ci(cmp.by_dt_h)

CSV.write("../res/17j_scores_frozen_by_model_horizon.csv", scores_f.by_model_h)
CSV.write("../res/17j_scores_frozen_by_model_date_horizon.csv", scores_f.by_model_dt_h)
CSV.write("../res/17j_compare_by_model_horizon.csv",
          leftjoin(cmp.by_h, select(ci_frame, Not(:ratio)); on = [:model, :horizon]))
CSV.write("../res/17j_compare_by_model_date_horizon.csv", cmp.by_dt_h)

println("===== log-scale WIS by model × horizon: conditional vs frozen =====")
show(select(cmp.by_h, :model, :horizon, :wis_conditional, :wis_frozen, :ratio), allrows = true); println()
println("\n===== paired ratio (mean over origins) with 90% origin-bootstrap band =====")
show(ci_frame, allrows = true); println()

## §3 The frozen scheme, 9j-style

These are 9j's own plotters, unchanged, run on the frozen scores (their titles still say "9j").
Read them against the matching `res/8j_*` / `res/9j_*` figures.

In [ ]:
save_show(plot_wis_by_horizon(scores_f, wins, cfg), "../res/17j_frozen_wis_by_horizon.png");
save_show(plot_wis_by_model_horizon(scores_f, model_labels, cfg), "../res/17j_frozen_wis_by_model_horizon.png");
save_show(plot_wis_by_horizon_over_time(scores_f, model_labels, model_cols, cfg),
          "../res/17j_frozen_wis_logscale_by_horizon_over_time.png");
save_show(plot_wis_diff_over_time(scores_f, model_labels, model_cols, cfg),
          "../res/17j_frozen_wis_diff_over_time.png");

In [ ]:
save_show(plot_rwis_bias_by_horizon(scores_f, model_labels, model_cols, cfg),
          "../res/17j_frozen_rwis_bias_by_horizon.png");
save_show(plot_rwis_by_age_horizon(scores_f, model_labels, model_cols, grid, cfg),
          "../res/17j_frozen_rwis_by_age_horizon.png");
save_show(plot_rwis_by_period(scores_f, model_labels, model_cols, cfg),
          "../res/17j_frozen_rwis_by_horizon_per_period.png");
save_show(plot_interval_coverage(scores_f, model_labels, model_cols, cfg),
          "../res/17j_frozen_coverage_50_90.png");
save_show(plot_forecast_panels(fc_f, wins, model_labels, model_cols, cfg; grid = grid, n = 15),
          "../res/17j_frozen_forecast_vs_observed_panels.png");

## §4 Conditional vs frozen

- **WIS by horizon**: one panel per model, solid = conditional, dashed = frozen.
- **Ratio** `mean WIS_frozen / mean WIS_conditional`: > 1 means the future contacts helped. The
  band is a 90% bootstrap over origins; the null line is the noise floor.
- **Per-origin paired difference** over time, by horizon: *when* did knowing the contacts matter?
- **Coverage** under each scheme, and **relative skill** vs `REF_MODEL` under each scheme. The
  latter asks whether the model ordering survives taking the future contacts away.
- **Fans** of one model under both schemes.

In [ ]:
save_show(plot_scheme_wis_by_horizon(cmp, model_labels, model_cols, cfg),
          "../res/17j_compare_wis_by_horizon.png");
save_show(plot_scheme_ratio_by_horizon(ci_frame, model_labels, model_cols, cfg),
          "../res/17j_compare_ratio_by_horizon.png");
save_show(plot_scheme_diff_over_time(cmp, model_labels, model_cols, cfg),
          "../res/17j_compare_diff_over_time.png");

In [ ]:
save_show(plot_scheme_coverage(cmp, model_labels, model_cols, cfg),
          "../res/17j_compare_coverage_50_90.png");
save_show(plot_scheme_rank_by_horizon(cmp, model_labels, model_cols, cfg),
          "../res/17j_compare_rwis_by_horizon.png");
for lbl in ("weighted-hweibull|neighbourhood", "unweighted-negbin|mean")
    save_show(plot_scheme_forecast_panels(fc_c, fc_f, wins, lbl, cfg; grid = grid, inf = inf, n = 9),
              "../res/17j_compare_forecast_panels_$(replace(lbl, "|" => "_")).png");
end

## §5 Verification

In [ ]:
# (1) Every h0 artefact really IS h0: z_c has n_fit−1 coordinates; one Cstar_end per Stage-1 draw.
shp = check_h0_artefact_shapes(wins, combos, cfg)
@assert all(shp.ok) "h0 artefacts with the wrong shape: $(shp[.!shp.ok, :file])"
println("(1) h0 artefact shapes OK: ", nrow(shp), " files")

# (2) Both schemes were scored against the SAME truth on the SAME units.
tc = assembly_truth(qall_c, grid; horizons = cfg.horizons)
tf = assembly_truth(qall_f, grid; horizons = cfg.horizons)
@assert tc.truth == tf.truth "conditional and frozen were scored against different truth"
println("(2) identical truth over ", tc.n_cells, " cells")

# (3) Re-scoring 9j's cache reproduces 9j's own written frame (when that is the full grid).
p9 = "../res/8j_scores_by_model_horizon.csv"
if isfile(p9) && Set(origins) == Set(asm_c0.origins)
    ref = @subset(CSV.read(p9, DataFrame), :scale .== "log")
    me  = @subset(scores_c.by_model_h, :scale .== "log")
    j   = innerjoin(select(ref, :model, :horizon, :wis => :wis_9j), select(me, :model, :horizon, :wis);
                    on = [:model, :horizon])
    d   = maximum(abs.(j.wis .- j.wis_9j))
    @assert nrow(j) == nrow(me) && d < 1e-9 "conditional re-score disagrees with 9j's CSV (max |Δ| = $d)"
    println("(3) conditional re-score reproduces ", p9, " (max |Δ| = ", d, ")")
else
    println("(3) skipped — ", isfile(p9) ? "this run is a subset of 9j's origins" : "no 9j CSV")
end

# (4) The noise floor: the null model's ratio should sit near 1 at every horizon.
nul = @subset(ci_frame, :model .== "no-contact|null")
println("(4) null-model frozen/conditional ratio by horizon: ", round.(nul.ratio; digits = 4),
        "  (90% band ", round.(nul.lo; digits = 3), " … ", round.(nul.hi; digits = 3), ")")
maximum(abs.(nul.ratio .- 1)) > 0.05 &&
    @warn "null-model ratio departs from 1 by > 5% — the comparison's noise floor is wider than expected"

# (5) Nothing outside 17j's own namespace was written by this notebook's run.
fresh = filter(f -> mtime(joinpath("../res", f)) > T_START && !startswith(f, "17j_"), readdir("../res"))
@assert isempty(fresh) "17j wrote outside its namespace: $(fresh)"
println("(5) only res/17j_* written by this run")

## §6 Six scenarios: variant-specific renewal × contact data (2026-10-21)

For each contact model `c` (the four ways + no-interaction) there are six scenarios. All are scored
on TOTAL infections by age, as above.

| # | scenario | series | from |
|---|---|---|---|
| 1 | no contact | `no-contact|null` | 9j conditional cache |
| 2 | contact (C\* to t₀+h) | `c` | 9j conditional cache |
| 3 | contact (frozen at t₀) | `c@frozen` | §1 frozen cache |
| 4 | variant | `no-contact|null-variant` | own cache `17j_assembly_variant_*` |
| 5 | variant + contact | `c-variant` | own cache `17j_assembly_variant_*` |
| 6 | variant + contact (frozen) | `c-variant@frozen` | own cache `17j_assembly_variant_frozen_*` |

**The variant model** (`VariantNGM`, `inst/3` §12; fitted in 9j §V):
- one renewal per COG-UK variant category, with γ_SAR per category; everything else is shared;
- categories are chosen at t₀ (genomes summed over the origin week and the 3 weeks before ≥ 50, or
  an origin-week share ≥ 1%); once included, a category is dropped only when its share is below
  0.1% in all 8 fit weeks (hysteresis); no variant data after t₀ is used;
- each variant is propagated by its own renewal, and the variants are summed for scoring.

Scenarios 1 and 4 use no contact data, so they are the same series in every panel. `@frozen` is a
series name only; it is never a path label.

⚠ **Still a pure reader.** The guard asserts every variant artefact before either assembly call,
and the 9j/§1 caches are read rather than rebuilt. Outputs are `res/17j_scenarios_*`.

In [ ]:
contact_labels  = filter(!=(NULL_LABEL), model_labels)        # the five contact models
vcombos         = [(dm, VariantNGM(nb)) for (dm, nb) in combos]
vcombos_contact = filter(c -> needs_stage1(c[1]), vcombos)
st  = scenario_table(contact_labels)
cog = load_cog_variant_counts()

# ⚠ GUARD — before ANY variant assembly call: a missing variant artefact would be FITTED on demand.
_tag = contacts_label(cfg)
_want = vcat(vec([stage2_path(dm, nb, w.origin, h; contacts = _tag, save_dir = "../dt_intermediate")
                  for (dm, nb) in vcombos, w in wins, h in cfg.horizons]),
             vec([stage2_path(dm, nb, w.origin, FROZEN_H; contacts = _tag, save_dir = "../dt_intermediate")
                  for (dm, nb) in vcombos_contact, w in wins]))
_miss = filter(!isfile, _want)
@assert isempty(_miss) "17j §6: $(length(_miss)) of $(length(_want)) variant artefacts missing, e.g. $(basename.(first(_miss, 3))) — run 9j §V first (17j would otherwise REFIT them)"
println("variant artefacts present: ", length(_want), " | scenarios × contact models = ", nrow(st),
        " (", length(unique(st.series)), " distinct series)")

# The first origin each category got its own renewal (for the over-time figure).
vset = variant_set_table(wins, cog, cfg)
entries = [(g, vset.origin[i]) for g in COG_GROUPS[2:end]
           for i in [findfirst(c -> g in split(c, "; "), vset.categories)] if i !== nothing]
println("first origin with its own renewal: ", join(["$(g) $(d)" for (g, d) in entries], ", "))

In [ ]:
REBUILD_VARIANT = false
asm_v  = assemble_or_load_forecasts(wins, vcombos, cfg; grid = grid, raw = raw,
                                    save_dir = "../dt_intermediate",
                                    cache_path = variant_cache_path(cfg; frozen = false),
                                    rebuild = REBUILD_VARIANT, variant_counts = cog)
asm_vf = assemble_or_load_forecasts(wins, vcombos_contact, cfg; grid = grid, raw = raw,
                                    save_dir = "../dt_intermediate",
                                    cache_path = variant_cache_path(cfg; frozen = true),
                                    rebuild = REBUILD_VARIANT, frozen = true, variant_counts = cog)
@assert isempty(asm_v.skipped) && isempty(asm_vf.skipped) "variant assembly skipped: $(asm_v.skipped) $(asm_vf.skipped)"

# One quantile frame and one fan store over every series (frozen ones renamed `<label>@frozen`).
qs = DataFrame[]; fc_s = Dict{Tuple{Date,String},Array{Float64,3}}()
for (q, fc, fz) in ((qall_c, fc_c, false), (qall_f, fc_f, true),
                    (asm_v.qall, asm_v.fc_store, false), (asm_vf.qall, asm_vf.fc_store, true))
    q2, fc2 = rename_series(q, fc, fz)
    push!(qs, q2); merge!(fc_s, fc2)
end
qall_s = @subset(vcat(qs...), in.(:model, Ref(Set(st.series))))
@assert Set(qall_s.model) == Set(st.series) "series without forecasts: $(setdiff(Set(st.series), Set(qall_s.model)))"
println("six-scenario quantile rows: ", nrow(qall_s), " over ", length(unique(qall_s.model)), " series")

### §6a Scores

In [ ]:
scores_s = score_wis(qall_s)
ratios   = scenario_pair_ratios(scores_s.by_model_dt_h, st)
smap     = unique(select(st, :series, :scenario, :label, :frozen))
CSV.write("../res/17j_scenarios_scores_by_model_horizon.csv",
          innerjoin(smap, scores_s.by_model_h; on = :series => :model))
CSV.write("../res/17j_scenarios_scores_by_model_date_horizon.csv",
          innerjoin(smap, scores_s.by_model_dt_h; on = :series => :model))
CSV.write("../res/17j_scenarios_compare_by_horizon.csv", ratios)

wide = unstack(select(innerjoin(@subset(scores_s.by_model_h, :scale .== WIS_SCALE), st; on = :model => :series),
                      :contact => :contact_model, :horizon, :scenario, :wis), :scenario, :wis)
println("===== mean $(WIS_SCALE)-scale WIS by contact model × horizon × scenario =====")
show(stdout, MIME("text/plain"), select(wide, :contact_model, :horizon, SCENARIO_ORDER...); allrows = true); println()
println("\n===== variant gain (mean WIS ratio, 90% origin bootstrap; < 1 ⇒ the variant split helped) =====")
show(stdout, MIME("text/plain"),
     select(@subset(ratios, :contrast .== "variant gain"), :contact, :num, :horizon, :ratio, :lo, :hi, :frac_num_better);
     allrows = true); println()

### §6b Figures

- **WIS by horizon**, one panel per contact model, with the six scenarios.
- **The same, before Omicron**: origins up to and including 2021-11-01.
- **Ranking before Omicron**: all 22 model × scenario series ranked by mean log WIS, one panel per horizon.
- **Relative WIS vs no contact**, with a 90% paired origin bootstrap.
- **Coverage**, 50% and 90%, and **bias** by horizon (scoringutils: > 0 over-, < 0 under-prediction),
  over all origins and again before Omicron (origins ≤ 2021-11-01).
- **Per-origin variant gain**: `(5) − (2)` per contact model and `(4) − (1)`. The dotted lines
  mark the origin at which each category first gets its own renewal.
- **WIS over time** for each contact model: one row per horizon, with the six scenarios' per-origin
  WIS (log) on the left and the cumulative WIS difference vs no contact on the right.
- **Forecast fans around each takeover** for each contact model: 15 panels, 5 origins centred on
  the first origin at which Alpha / Delta / Omicron BA.1 holds ≥ 50% of genomes, with all six
  scenarios overlaid. The y-axis is capped at 3× the observed maximum, and fans above the cap are
  counted in the panel title.
- **WIS by horizon per pandemic period** for each contact model: absolute mean log WIS, with 9j's
  periods plus "before Lockdown 2" and "after Opening up" (the Omicron origins), which 9j drops.
- **Age-stratified forecast at origin 2021-05-10** for each contact model: one panel per CIS age bin
  plus all ages, with the six scenarios overlaid.

In [ ]:
save_show(plot_scenario_wis_by_horizon(scores_s.by_model_h, st, cfg),
          "../res/17j_scenarios_wis_by_horizon.png");
save_show(plot_scenario_rel_wis(ratios, cfg),
          "../res/17j_scenarios_rel_wis_vs_no_contact.png");
save_show(plot_scenario_coverage(scores_s.by_model_h, st, cfg),
          "../res/17j_scenarios_coverage_50_90.png");
save_show(plot_variant_gain_over_time(scores_s.by_model_dt_h, st, cfg; entries = entries),
          "../res/17j_scenarios_variant_gain_over_time.png");

In [ ]:
# BIAS by horizon over all origins (scoringutils: > 0 over-, < 0 under-prediction), the companion of the
# coverage figure above.
save_show(plot_scenario_bias(scores_s.by_model_h, st, cfg; subtitle = " — all origins"),
          "../res/17j_scenarios_bias_by_horizon.png");

In [ ]:
# WIS by horizon BEFORE the Omicron period: origins up to and including 2021-11-01 (in practice the last
# origin ≤ that date, 2021-10-31), one panel per contact model, six scenarios. Same plotter as above on
# the restricted means; checked first to reproduce the all-origin `by_model_h` when unrestricted
# (RELATIVE tolerance: natural-scale WIS reaches ~2e7, where the two summations differ by ~4e-9).
PRE_OMICRON_END = Date(2021, 11, 1)
let full = scenario_wis_by_horizon_until(scores_s.by_model_dt_h, maximum(origins)),
    j = innerjoin(full, select(scores_s.by_model_h, :model, :horizon, :scale, :wis => :wis0); on = [:model, :horizon, :scale])
    @assert nrow(j) == nrow(scores_s.by_model_h) && maximum(abs.(j.wis .- j.wis0) ./ abs.(j.wis0)) < 1e-12 "per-origin mean ≠ by_model_h"
end
wis_pre = scenario_wis_by_horizon_until(scores_s.by_model_dt_h, PRE_OMICRON_END)
n_pre = maximum(wis_pre.n_origins)
CSV.write("../res/17j_scenarios_wis_by_horizon_pre_omicron.csv",
          innerjoin(unique(select(st, :series, :scenario, :label, :frozen)), wis_pre; on = :series => :model))
println("pre-Omicron origins: ", n_pre, " (", first(origins), " … ", maximum(filter(<=(PRE_OMICRON_END), origins)), ")")
save_show(plot_scenario_wis_by_horizon(wis_pre, st, cfg;
                                       subtitle = " — origins ≤ $(PRE_OMICRON_END) (n = $(n_pre), before Omicron)"),
          "../res/17j_scenarios_wis_by_horizon_pre_omicron.png");

In [ ]:
# RANKING of every model × scenario series (6 models: the 5 contact models × 4 contact scenarios + the
# no-contact model's 2 = 22 distinct series) by mean log WIS over the same pre-Omicron origins, by horizon.
rank_pre = scenario_rank_frame(wis_pre, st)
CSV.write("../res/17j_scenarios_rank_pre_omicron.csv", rank_pre)
show(stdout, MIME("text/plain"), unstack(select(rank_pre, :series, :horizon, :rank), :horizon, :rank; renamecols = h -> "rank_h$(h)");
     allrows = true, truncate = 60); println()
save_show(plot_scenario_rank_by_horizon(rank_pre, cfg; title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/17j_scenarios_rank_pre_omicron.png");

In [ ]:
# COVERAGE and BIAS before Omicron (origins ≤ PRE_OMICRON_END), from the same restricted means as the
# pre-Omicron WIS figure (`wis_pre`).
save_show(plot_scenario_coverage(wis_pre, st, cfg; subtitle = " — origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/17j_scenarios_coverage_50_90_pre_omicron.png");
save_show(plot_scenario_bias(wis_pre, st, cfg; subtitle = " — origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/17j_scenarios_bias_by_horizon_pre_omicron.png");

In [ ]:
# 9j-style forecast panels around each takeover, ONE figure per contact model: 15 panels = 5 origins
# centred on the first origin at which Alpha / Delta / Omicron BA.1 holds ≥ 50% of genomes (one
# column per takeover), all six scenarios overlaid (frozen dashed). Fans far above the data are
# counted in the panel title (y-axis capped at 3× the observed maximum).
tk = takeover_origins(wins, cog, cfg)
println("takeover origins: ", join(["$(g): $(first(o))…$(last(o)) (≥50% at $(t))" for (g, o, t) in tk], "; "))
for c in contact_labels
    save_show(plot_scenario_forecast_panels(fc_s, wins, st, c, cfg; groups = tk, grid = grid, inf = inf),
              "../res/17j_scenarios_forecast_takeovers_$(replace(c, "|" => "_")).png");
end

In [ ]:
# WIS over time, ONE figure per contact model: rows = horizons; left = each scenario's per-origin WIS
# (log), right = the running sum of WIS(scenario) − WIS(no contact) over origins (< 0 ⇒ better than no
# contact cumulatively; the endpoint is the whole-period difference). Dotted = a variant category enters.
for c in contact_labels
    save_show(plot_scenario_wis_over_time(scores_s.by_model_dt_h, st, c, cfg; entries = entries),
              "../res/17j_scenarios_wis_over_time_$(replace(c, "|" => "_")).png");
end

In [ ]:
# ABSOLUTE mean WIS (log) by horizon per pandemic period, ONE figure per contact model, six scenarios.
# Periods are 9j's (Munday 2023 Table 2) plus two explicit buckets 9j drops: "before Lockdown 2" (3
# origins) and "after Opening up" (5 origins, 2021-11-28 on — the Omicron takeover).
wis_period = scenario_wis_by_period(scores_s.by_model_dt_h, st)
CSV.write("../res/17j_scenarios_wis_by_period.csv", wis_period)
println("origins per period: ", join(["$(r.period) $(r.n)" for r in eachrow(
    combine(groupby(@subset(wis_period, :horizon .== 1), :period), :n_origins => maximum => :n))], ", "))
for c in contact_labels
    save_show(plot_scenario_wis_by_period(wis_period, c, cfg),
              "../res/17j_scenarios_wis_by_period_$(replace(c, "|" => "_")).png");
end

In [ ]:
# AGE-STRATIFIED forecast at origin 2021-05-10 (the Delta takeover; 10j's origin), ONE figure per contact
# model: a panel per CIS age bin + all ages, six scenarios overlaid. y capped at 3× the observed maximum,
# fans above it counted in the panel title.
AGE_ORIGIN = Date(2021, 5, 10)
@assert AGE_ORIGIN in origins
for c in contact_labels
    save_show(plot_scenario_age_fans(fc_s, wins, st, c, AGE_ORIGIN, cfg; grid = grid, inf = inf),
              "../res/17j_scenarios_age_fans_$(AGE_ORIGIN)_$(replace(c, "|" => "_")).png");
end

In [ ]:
# Fig 4 of Bracher et al. 2021 (ref/s41467-021-25207-0.pdf) for ONE contact model: point forecasts (median)
# h1–4 of the six scenarios from one origin (dotted, open markers; §6 colours, diamond = frozen), over the
# observed series; dashed line = origin. Total: one panel per origin. By age: ONE
# figure per origin — all ages on top, the 7 CIS bins below. A panel's median is of the per-draw SUM over
# its ages, so the age panels do not add up to the total.
FIG4_CONTACT = "weighted-hweibull|neighbourhood"
FIG4_ORIGINS = [Date(2020, 12, 7), Date(2021, 5, 10)]
FIG4_TAGS    = Dict(Date(2020, 12, 7) => "Alpha", Date(2021, 5, 10) => "Delta")
@assert all(in(origins), FIG4_ORIGINS) && FIG4_CONTACT in contact_labels
fig4_stem = "../res/17j_scenarios_point_forecasts"
fig4_c    = replace(FIG4_CONTACT, "|" => "_")
r = plot_scenario_point_forecasts(fc_s, st, FIG4_CONTACT, FIG4_ORIGINS, cfg; grid, inf, tags = FIG4_TAGS)
CSV.write("$(fig4_stem)_total_$(fig4_c).csv", r.frame)
save_show(r.fig, "$(fig4_stem)_total_$(fig4_c).png");
fig4_age = map(FIG4_ORIGINS) do o
    ra = plot_scenario_point_forecasts_by_age(fc_s, st, FIG4_CONTACT, o, cfg; grid, inf, tag = FIG4_TAGS[o])
    save_show(ra.fig, "$(fig4_stem)_age_$(o)_$(fig4_c).png");
    ra.frame
end
CSV.write("$(fig4_stem)_age_$(fig4_c).csv", vcat(fig4_age...))

### §6c Verification

In [ ]:
# (6) Every series was scored against the SAME truth (assembly_truth asserts `observed` agrees
#     across models), and on the same cells as §1–§5.
ts = assembly_truth(qall_s, grid; horizons = cfg.horizons)
@assert ts.truth == tc.truth "the six-scenario frame was scored against a different truth"
println("(6) identical truth over ", ts.n_cells, " cells for all ", length(unique(qall_s.model)), " series")

# (7) Every series covers every origin × horizon × age bin.
nunit = combine(groupby(unique(select(qall_s, :model, :forecast_date, :horizon, :age_group)), :model), nrow => :n)
@assert all(nunit.n .== length(wins) * length(cfg.horizons) * length(grid.LAB)) "series with missing units: $(nunit[nunit.n .!= length(wins) * length(cfg.horizons) * length(grid.LAB), :])"
println("(7) every series: ", length(wins), " origins × ", length(cfg.horizons), " h × ", length(grid.LAB), " ages")

# (8) The non-variant series re-score to §2's frames (scoring one model does not depend on the others).
let me = @subset(scores_s.by_model_h, :scale .== WIS_SCALE)
    d = 0.0
    for (sc, fz) in ((scores_c, false), (scores_f, true))
        r = @subset(sc.by_model_h, :scale .== WIS_SCALE)
        r = transform(r, :model => ByRow(m -> series_name(m, fz)) => :model)
        j = innerjoin(select(r, :model, :horizon, :wis => :wis0), select(me, :model, :horizon, :wis); on = [:model, :horizon])
        d = max(d, maximum(abs.(j.wis .- j.wis0)))
    end
    @assert d < 1e-9 "six-scenario re-score of the existing series disagrees with §2 (max |Δ| = $d)"
    println("(8) existing series reproduce §2's scores (max |Δ| = ", d, ")")
end

# (9) Nothing outside 17j's namespace was written.
fresh = filter(f -> mtime(joinpath("../res", f)) > T_START && !startswith(f, "17j_"), readdir("../res"))
@assert isempty(fresh) "17j wrote outside its namespace: $(fresh)"
println("(9) only res/17j_* written by this run")